In [1]:
import sys
sys.path.append('../')
import numpy as np
import scqubits.settings as settings
from joblib import Parallel, delayed
import utils_2Q_gate_zp as ut
import qutip as qt
settings.OVERLAP_THRESHOLD = 0.3

import pandas as pd
import scipy.sparse as ssp

## cz generate eval

In [ ]:
test = True
if test:
    truc1, truc_tot, charge_pick = 11, 15, True       
else:
    truc1, truc_tot, charge_pick = 300, 2000, True   
g=0.030961990356445312
print(f'truc1, truc_tot, charge_pick = {truc1}, {truc_tot}, {charge_pick}')
print(f'g={g}')

eval0, eval1, n_theta0, n_theta1 = ut.load_1q_data_for_2q(truc1)
##############################################################################################
if charge_pick:
    hspace_0 = ut.get_truncated_subspace_xgate(n_theta0, truc1)
    hspace_1 = ut.get_truncated_subspace_xgate(n_theta1, truc1)
    n_theta0 = ut.truncate_2(n_theta0, hspace_0)
    n_theta1 = ut.truncate_2(n_theta1, hspace_1)
    eval0 = eval0[hspace_0]
    eval1 = eval1[hspace_1]

truc1, truc_tot, charge_pick = 11, 15, True
g=0.030961990356445312


In [ ]:
##############################################################################################
###  Compute eigenvalues and eigenvectors for coupling H
print(f'len(hspace_0)={len(hspace_0)}, len(hspace_1)={len(hspace_1)}')
Hint = qt.tensor(qt.Qobj(n_theta0) , qt.Qobj(n_theta1))
H_bare = (  qt.tensor(qt.Qobj(np.diag(eval0)),  qt.identity(len(hspace_1)))
        +  qt.tensor(qt.identity(len(hspace_0)),  qt.Qobj(np.diag(eval1))) )
Htot = g* Hint + H_bare

### the one-line code below takes time when truc1 is large
k = Htot.shape[0] - 1
if truc_tot != None:
    k = truc_tot
eval_tot, eket_tot = ssp.linalg.eigsh(Htot.data, k=k, which='SA', tol=1.e-10)

sorted_idx_tot = np.argsort(eval_tot)
eval_tot = eval_tot[sorted_idx_tot]
eval_tot = eval_tot - eval_tot[0]
eket_tot = ssp.csr_matrix([eket_tot[:,idx] for idx in sorted_idx_tot])

folder_save = f'data/3ncut_two_zeropi/truc1={truc1}_truc2={truc_tot}_pick={charge_pick}/'
os.mkdir(folder_save)
pd.DataFrame(eval_tot).to_csv(folder_save+ 'eval_tot.txt', sep=',', index=False, header=True)
np.save(folder_save+'eket_tot.npy', eket_tot.toarray())
print(f'np.shape(eket_tot)={np.shape(eket_tot)}')

## cz generate nop

In [6]:
truc1, truc_tot, charge_pick = 150, 2000, True

if charge_pick:
    folder = f'data/3ncut_two_zeropi/truc1={truc1}_truc2={truc_tot}_pick={charge_pick}/'
    hspace_0 = pd.read_csv(folder+ 'hspace_0.txt').to_numpy().flatten().tolist()
    hspace_1 = pd.read_csv(folder+ 'hspace_1.txt').to_numpy().flatten().tolist()
else:
    hspace_0 = np.arange(truc1).tolist()
    hspace_1 = np.arange(truc1).tolist()

In [2]:
truc1, truc_tot, charge_pick = 11, 15, True

if charge_pick:
    folder = f'data/3ncut_two_zeropi/truc1={truc1}_truc2={truc_tot}_pick={charge_pick}/'
    hspace_0 = pd.read_csv(folder+ 'hspace_0.txt').to_numpy().flatten().tolist()
    hspace_1 = pd.read_csv(folder+ 'hspace_1.txt').to_numpy().flatten().tolist()
else:
    hspace_0 = np.arange(truc1).tolist()
    hspace_1 = np.arange(truc1).tolist()

n0 = len(hspace_0)
n1 = len(hspace_1)

folder = f'../../data/3ncut_two_zeropi/truc1=500/'
n_theta0 = np.load(folder+'n_theta0.npy')
n_theta1 = np.load(folder+'n_theta1.npy')

n_theta0 = ut.truncate_2(n_theta0, hspace_0)
n_theta1 = ut.truncate_2(n_theta1, hspace_1)

folder_save = f'data/3ncut_two_zeropi/truc1={truc1}_truc2={truc_tot}_pick={charge_pick}/'
eval_tot = pd.read_csv(folder_save+ 'eval_tot.txt').to_numpy().flatten()
eket_tot = ssp.csr_matrix(np.load(folder_save+ 'eket_tot.npy'))


In [3]:
###  Get wavefunction overlap for the truncated dressed states
bare_state = [[qt.tensor(qt.basis(n0, i), qt.basis(n1, j))
                        for j in range(n1)]
                            for i in range(n0)]
arg = [bare_state, n0, n1]

print("ut.find_overlap..... Time:")
ut.print_time()
result = Parallel(n_jobs=100)(delayed(ut.find_overlap)(i, *arg) for i in eket_tot)
top_index = [result[i][0] for i in range(eval_tot.shape[0])]
top_overlap = [result[i][1] for i in range(eval_tot.shape[0])]
np.save(folder_save+f'top_index.npy', top_index)
np.save(folder_save+f'top_overlap.npy', top_overlap)
print("pd.DataFrame(top_overlap)..... Time:")
ut.print_time()
##############################################################################################
### Get the dressed states index
hspace_full = ut.get_dressed_states_index(top_index, hspace_0, hspace_1)

ut.find_overlap..... Time:

Current China Time: 2025-08-04 10:20:22.752585+08:00


pd.DataFrame(top_overlap)..... Time:

Current China Time: 2025-08-04 10:20:25.760491+08:00


In [5]:
hspace_full

['0-0',
 '0-1',
 '1-0',
 '0-2',
 '2-0',
 '0-4',
 '4-0',
 '1-1',
 '0-5',
 '2-1',
 '5-0',
 '1-2',
 '0-8',
 '2-2',
 '8-0']

In [4]:
=

SyntaxError: invalid syntax (1763773627.py, line 1)

In [ ]:
n_theta0_dress = ssp.kron(n_theta0, ssp.identity(n1))
n_theta1_dress = ssp.kron(ssp.identity(n0), n_theta1)
n_theta0_dress = (eket_tot @ n_theta0_dress @ eket_tot.conj().T).todense()
n_theta1_dress = (eket_tot @ n_theta1_dress @ eket_tot.conj().T).todense()
print("folder_save.... Time:")
ut.print_time()

pd.DataFrame(hspace_full).to_csv(folder_save+ f'hspace_full.txt', sep=',', index=False, header=True)


pd.DataFrame(top_overlap)..... Time:

Current China Time: 2025-08-04 09:31:07.417948+08:00
folder_save.... Time:

Current China Time: 2025-08-04 09:31:07.422498+08:00


In [ ]:
np.save(folder_save+f'n_theta0_dress.npy', n_theta0_dress)
np.save(folder_save+f'n_theta1_dress.npy', n_theta1_dress)
print(f'len(hspace_full) = {len(hspace_full)}')
print(f'np.shape(n_theta0_dress) = {np.shape(n_theta0_dress)}')
print(f'np.shape(n_theta1_dress) = {np.shape(n_theta1_dress)}')

len(hspace_full) = 15
np.shape(n_theta0_dress) = (15, 15)
np.shape(n_theta1_dress) = (15, 15)


In [ ]:
=

## cz noise model

In [ ]:
n_truc = 100 # [ 20, 40, 50, 60, 80, 100, 150, 200, ] # [100, 200, 300, 400, 500] # List of truncation sizes to test
filter_ratio = 0.3

cz_run = True # True  # whether to use CZ gate or CNOT gate
import_2000_states = False # whether to import 2000 or 1000 states Hamiltonian
truc_one_qubit = 300 # truncation for single zero pi
n_full = 1000 # don't change this value, If import_2000 = True, n_full=2000, else 1000

# 'short_path', 'all_path', 'hand_pick', 
# 'cz_short_500_500_detune0', 'cz_short_500_500_detune1' 
use_truc_model, truc_model_name = True, 'cz_short_500_500_detune1'

# below sets whether to calculate noisy fidelity, they should not be true at the same time to avoid error
calculate_ideal, calculate_noise = False, True # True, False #   
t1_tphi_other = 3 # μs
tg_list = [16] # [2, 9, 16, 23, 30] # Select the first row for testing
max_step_ideal, max_step_noisy = 1e-3, 1e-3 # Set max_step to 0 for parallel execution
num_cpus, n_job = 16, len(tg_list) # Number of CPUs and jobs for parallel processing

[hspace_full, eket_tot, eval_tot, n_theta0_dress, 
    n_theta1_dress, hspace_0, hspace_1, logi_state
    ] = ut.load_qubit_data_2q(n_full, import_2000_states, truc_one_qubit)
dim_0 = len(hspace_0)
dim_1 = len(hspace_1) 
params = ut.load_drive_params_2q(cz_run)[tg_list, ]  # [1::4,] # Load pulse parameters from CSV

if cz_run: # CZ
    drive_term = n_theta1_dress
    W_20_50 = ( eval_tot[hspace_full.index('5-0')] - 
                eval_tot[hspace_full.index('2-0')] )
    print(f'Using CZ gate with W_20_50 = {W_20_50}')
else: # CNOT     
    drive_term = n_theta0_dress
    mid_state = '8-2'
    idx_0 = hspace_full.index('0-2')
    idx_1 = hspace_full.index('2-2')
    idx_2 = hspace_full.index(mid_state)
    W_0_2 = eval_tot[idx_2] - eval_tot[idx_0]
    W_1_2 = eval_tot[idx_2] - eval_tot[idx_1]
    print(f'Using CNOT gate with W_0_2 = {W_0_2}, W_1_2 = {W_1_2}')
    print('\nmid_state = ', mid_state)
print(f"calculate_ideal = {calculate_ideal}, calculate_noise = {calculate_noise}")
option_ideal, option_noisy = ut.get_qutip_options(max_step_ideal, max_step_noisy) 
print(f'filter_ratio = {filter_ratio}')     
print(f'get_hamiltonian_given_state_list = {use_truc_model}')
print(f"t1_tphi_other = {t1_tphi_other}, import_2000={import_2000_states}")
print('truc_full=', n_full )
print('num_cpus=', num_cpus, ', n_job=', n_job)
ut.print_data(f'params', params.tolist(), num_each_row=1)    


Using CZ gate with W_20_50 = 15.982875383444132
calculate_ideal = False, calculate_noise = True
Ideal: max_step = 0.001, nsteps = 1000.0
Noisy: max_step = 0.001, nsteps = 1000.0
filter_ratio = 0.3
get_hamiltonian_given_state_list = True
t1_tphi_other = 3, import_2000=False
truc_full= 1000
num_cpus= 16 , n_job= 1

params = np.array([
[116.077689, 0.013154, 0.020213] ,
])


In [ ]:
if use_truc_model:
    if cz_run:
        hspace_select = ut.cz_truc_model[truc_model_name][:n_truc]
    else:
        hspace_select = ut.cnot_truc_model[truc_model_name][:n_truc]
else:
    hspace_select = hspace_full[:n_truc]
index_select = [hspace_full.index(i) for i in hspace_select]
H_drive_select, eket_truc = ut.build_hamiltonian_2q(cz_run, index_select, eval_tot, 
                                                    eket_tot, drive_term)
logi_idx_select = [hspace_select.index(i) for i in logi_state]

ut.print_data(f'hspace_select (len={len(hspace_select)})', 
                hspace_select, num_each_row=10)

ut.print_data(f'index_select (len={n_truc})', index_select, num_each_row=10)



hspace_select (len=100) = np.array([
'0-0', '2-2', '0-2', '2-0', '5-0', '0-1', '2-1', '0-5', '2-5', '1-0' ,
'5-2', '1-2', '5-1', '0-4', '2-4', '1-1', '0-9', '2-9', '4-0', '5-5' ,
'1-5', '4-2', '9-0', '0-12', '5-4', '2-12', '0-8', '2-8', '9-2', '4-1' ,
'1-9', '9-1', '0-16', '2-16', '1-4', '8-0', '4-5', '0-13', '2-13', '0-18' ,
'2-18', '1-12', '9-5', '9-4', '2-21', '5-9', '4-4', '0-21', '1-16', '0-26' ,
'5-8', '18-0', '1-8', '5-12', '15-4', '2-26', '8-2', '0-24', '2-24', '2-45' ,
'0-20', '0-45', '2-20', '2-39', '0-39', '8-12', '8-1', '0-25', '1-18', '2-25' ,
'12-0', '0-33', '2-33', '0-34', '4-9', '2-34', '5-16', '0-36', '0-52', '2-52' ,
'2-36', '2-59', '0-59', '2-65', '0-65', '15-0', '15-1', '0-42', '2-42', '0-30' ,
'9-8', '2-30', '0-55', '2-55', '12-2', '1-20', '1-21', '5-13', '8-5', '13-0' ,
])

index_select (len=100) = np.array([
0, 13, 3, 4, 10, 1, 9, 8, 24, 2 ,
27, 11, 21, 5, 18, 7, 17, 41, 6, 48 ,
20, 22, 19, 23, 38, 56, 12, 34, 46, 16 ,
35, 36, 29, 67, 15, 14, 39, 28, 65, 40 ,
89

In [ ]:
[n_theta0, n_theta1, gamma_dephase_02_q0, gamma_dephase_02_q1
] = ut.load_noise_data_2q(t1_tphi_other) 

Gamma = 1 / 1e3 / t1_tphi_other
Gamma_decay_q0 = Gamma / (n_theta0[4,8]**2)
Gamma_decay_q1 = Gamma / (n_theta1[4,8]**2)


In [ ]:
n_theta1[4,8]**2

(4.359655343970712+0j)

In [ ]:
np.shape(n_theta0)

(500, 500)

In [ ]:
hspace_0

array([  0,   1,   2,   4,   5,   8,   9,  12,  13,  15,  18,  20,  22,
        24,  25,  26,  28,  30,  33,  34,  35,  37,  39,  41,  44,  45,
        46,  50,  51,  54,  56,  58,  59,  60,  64,  65,  66,  69,  70,
        74,  75,  77,  79,  81,  84,  85,  86,  89,  90,  92,  93,  98,
        99, 100, 101, 102, 105, 106, 109, 110, 111, 115, 116, 121, 123,
       124, 127, 128, 129, 131, 132, 135, 136, 138, 140, 141, 144, 145,
       147, 150, 151, 154, 157, 160, 161, 162, 165, 167, 169, 170, 173,
       174, 175, 177, 179, 180, 183, 185, 188, 189, 194, 195, 198, 199,
       200, 201, 204, 205, 207, 208, 210, 213, 214, 216, 218, 222, 223,
       224, 225, 228, 229, 234, 235, 238, 241, 244, 247, 248, 249, 250,
       251, 252, 254, 255, 260, 261, 262, 263, 264, 265, 269, 270, 271,
       275, 277, 281, 282, 286, 288, 289, 293, 294, 295, 299])

In [ ]:
hspace, n_theta = hspace_0, n_theta0
low_states=50

low = [s for s in hspace if s < low_states]
n_theta_low = ut.truncate_2(n_theta, low)
n_max = np.abs(n_theta_low.data).max()

n_theta_trunc = ut.truncate_2(n_theta, hspace)
def ratio(sj):
    return filter_ratio if sj < low_states else 10 * filter_ratio
transition = [
    (i, j)
    for i, si in enumerate(hspace)
    for j, sj in enumerate(hspace)
    if i < j and abs(n_theta_trunc[i, j]) > n_max * ratio(sj)
]

In [ ]:
print(transition)

[(0, 1), (1, 3), (2, 4), (3, 5), (4, 6), (5, 7), (6, 9), (6, 10), (7, 9), (7, 11), (8, 10), (9, 14), (10, 12), (10, 17), (11, 12), (11, 13), (11, 14), (12, 20), (13, 15), (14, 15), (14, 23), (16, 22), (17, 18), (17, 20), (18, 19), (19, 20), (20, 21), (20, 24), (23, 24), (23, 26), (24, 25), (25, 26)]


In [ ]:
n_theta_trunc[6,10]

(-1.31813208+0j)

In [ ]:
n_theta_trunc

Quantum object: dims = [[154], [154]], shape = (154, 154), type = oper, isherm = True
Qobj data =
[[ 0.00000000e+00 -1.35218494e+00  0.00000000e+00 ... -4.01500000e-05
   0.00000000e+00 -1.07000000e-05]
 [-1.35218494e+00  0.00000000e+00 -1.03945100e-02 ...  0.00000000e+00
  -2.85100000e-05  0.00000000e+00]
 [ 0.00000000e+00 -1.03945100e-02  0.00000000e+00 ... -3.23800000e-05
   0.00000000e+00  4.00000000e-07]
 ...
 [-4.01500000e-05  0.00000000e+00 -3.23800000e-05 ...  0.00000000e+00
  -1.79172481e+00  0.00000000e+00]
 [ 0.00000000e+00 -2.85100000e-05  0.00000000e+00 ... -1.79172481e+00
   0.00000000e+00  1.02396859e+00]
 [-1.07000000e-05  0.00000000e+00  4.00000000e-07 ...  0.00000000e+00
   1.02396859e+00  0.00000000e+00]]

In [ ]:
transition_a, n_theta0_trunc = ut.get_transitions_for_collapse(hspace_0, n_theta0, 
                                                            filter_ratio=filter_ratio)
transition_b, n_theta1_trunc = ut.get_transitions_for_collapse(hspace_1, n_theta1, 
                                                            filter_ratio=filter_ratio)

In [ ]:
eket_truc

<100x23870 sparse matrix of type '<class 'numpy.complex128'>'
	with 2387000 stored elements in Compressed Sparse Row format>

In [ ]:
c_op_list_new = [c_op_list[i]* 10 for i, _ in enumerate(c_op_list)]

In [ ]:
c_op_list_new[0]

Quantum object: dims = [[100], [100]], shape = (100, 100), type = oper, isherm = True
Qobj data =
[[ 8.86825378e-08+0.00000000e+00j  2.92425341e-11+1.30886350e-11j
  -1.35872276e-09-3.62680732e-09j ...  0.00000000e+00+0.00000000e+00j
  -1.22586722e-09-4.75989911e-10j  2.16253040e-08-1.07323485e-08j]
 [ 2.92425341e-11-1.30886350e-11j  5.94244314e-11+0.00000000e+00j
   1.60936679e-09+1.62912816e-09j ...  0.00000000e+00+0.00000000e+00j
   1.36025939e-09-6.87209814e-11j -4.96968016e-11+6.03028010e-11j]
 [-1.35872276e-09+3.62680732e-09j  1.60936679e-09-1.62912816e-09j
   8.86439064e-08+0.00000000e+00j ...  0.00000000e+00+0.00000000e+00j
   3.93672294e-08-4.40945003e-08j  8.94095235e-10+8.71600095e-09j]
 ...
 [ 0.00000000e+00+0.00000000e+00j  0.00000000e+00+0.00000000e+00j
   0.00000000e+00+0.00000000e+00j ...  3.62841599e-09+0.00000000e+00j
   0.00000000e+00+0.00000000e+00j  0.00000000e+00+0.00000000e+00j]
 [-1.22586722e-09+4.75989911e-10j  1.36025939e-09+6.87209814e-11j
   3.93672294e-08+4

In [ ]:
c_op_list[0]

Quantum object: dims = [[100], [100]], shape = (100, 100), type = oper, isherm = True
Qobj data =
[[ 8.86825378e-09-3.23102407e-27j  2.92425341e-12+1.30886350e-12j
  -1.35872276e-10-3.62680732e-10j ... -9.34659035e-22-1.89142411e-21j
  -1.22586722e-10-4.75989911e-11j  2.16253040e-09-1.07323485e-09j]
 [ 2.92425341e-12-1.30886350e-12j  5.94244314e-12-4.04686336e-28j
   1.60936679e-10+1.62912816e-10j ...  1.03025987e-22-1.98320757e-22j
   1.36025939e-10-6.87209814e-12j -4.96968016e-12+6.03028010e-12j]
 [-1.35872276e-10+3.62680732e-10j  1.60936679e-10-1.62912816e-10j
   8.86439064e-09-3.18710036e-27j ...  4.10928822e-24-4.28978026e-22j
   3.93672294e-09-4.40945003e-09j  8.94095235e-11+8.71600095e-10j]
 ...
 [-9.34659035e-22+1.89142411e-21j  1.03025987e-22+1.98320757e-22j
   4.10928822e-24+4.28978026e-22j ...  3.62841599e-10-1.71919178e-27j
   8.56839472e-22-2.45147385e-20j -1.15360777e-22+3.08029508e-21j]
 [-1.22586722e-10+4.75989911e-11j  1.36025939e-10+6.87209814e-12j
   3.93672294e-09+4

In [ ]:
c_op_list = ut.construct_c_ops_2q(dim_0, dim_1, n_theta0_trunc, n_theta1_trunc, 
                                    gamma_dephase_02_q0, gamma_dephase_02_q1, eket_truc, 
                                    Gamma_decay_q0, Gamma_decay_q1, 
                                    transition_a, transition_b, 
                                    apply_decay=False, apply_dephase=False)    
print(f'np.shape(c_op_list) = {np.shape(c_op_list)}')     

len(jump_tphi_a) = 0, len(jump_tphi_b) = 0
len(jump_t1_a) = 0, len(jump_t1_b) = 0
np.shape(c_op_list) = (0,)


In [ ]:
c_op_list = ut.construct_c_ops_2q(dim_0, dim_1, n_theta0_trunc, n_theta1_trunc, 
                                    gamma_dephase_02_q0, gamma_dephase_02_q1, eket_truc, 
                                    Gamma_decay_q0, Gamma_decay_q1, 
                                    transition_a, transition_b, 
                                    apply_decay=True, apply_dephase=False)    
print(f'np.shape(c_op_list) = {np.shape(c_op_list)}')   


len(jump_tphi_a) = 0, len(jump_tphi_b) = 0
len(jump_t1_a) = 32, len(jump_t1_b) = 28
np.shape(c_op_list) = (60, 100, 100)


In [ ]:
c_op_list = ut.construct_c_ops_2q(dim_0, dim_1, n_theta0_trunc, n_theta1_trunc, 
                                    gamma_dephase_02_q0, gamma_dephase_02_q1, eket_truc, 
                                    Gamma_decay_q0, Gamma_decay_q1, 
                                    transition_a, transition_b, 
                                    apply_decay=False, apply_dephase=True)    
print(f'np.shape(c_op_list) = {np.shape(c_op_list)}')     


len(jump_tphi_a) = 153, len(jump_tphi_b) = 154
len(jump_t1_a) = 0, len(jump_t1_b) = 0
np.shape(c_op_list) = (307, 100, 100)


In [ ]:
c_op_list = ut.construct_c_ops_2q(dim_0, dim_1, n_theta0_trunc, n_theta1_trunc, 
                                    gamma_dephase_02_q0, gamma_dephase_02_q1, eket_truc, 
                                    Gamma_decay_q0, Gamma_decay_q1, 
                                    transition_a, transition_b, 
                                    apply_decay=True, apply_dephase=True)    
print(f'np.shape(c_op_list) = {np.shape(c_op_list)}')     


len(jump_tphi_a) = 153, len(jump_tphi_b) = 154
len(jump_t1_a) = 32, len(jump_t1_b) = 28
np.shape(c_op_list) = (367, 100, 100)


In [ ]:
max_value = []
for i,_ in enumerate(c_op_list):
    max_value.append( abs(np.max(c_op_list[i].full())))
    print(f'c_ops[{i}] = {abs(  np.max(c_op_list[i].full()) ):.8f}') # for debugging
print(f'dephase_max = {np.max(max_value[:307]):.8f}, decay_max = {np.max(max_value[307:]):.8f}' )
print(f'dephase_max_idx = {np.argmax(max_value[:307])}, decay_max = {np.argmax(max_value[307:])}' )

c_ops[0] = 0.00007064
c_ops[1] = 0.02581677
c_ops[2] = 0.00010026
c_ops[3] = 0.02239597
c_ops[4] = 0.00018996
c_ops[5] = 0.00898726
c_ops[6] = 0.00046218
c_ops[7] = 0.01785183
c_ops[8] = 0.00649358
c_ops[9] = 0.00160232
c_ops[10] = 0.00001173
c_ops[11] = 0.00005617
c_ops[12] = 0.00000078
c_ops[13] = 0.00001787
c_ops[14] = 0.00000006
c_ops[15] = 0.00000022
c_ops[16] = 0.00000036
c_ops[17] = 0.00001341
c_ops[18] = 0.00000025
c_ops[19] = 0.00000180
c_ops[20] = 0.00000001
c_ops[21] = 0.00000009
c_ops[22] = 0.00000004
c_ops[23] = 0.00000000
c_ops[24] = 0.00000000
c_ops[25] = 0.00000001
c_ops[26] = 0.00000000
c_ops[27] = 0.00000000
c_ops[28] = 0.00000000
c_ops[29] = 0.00000000
c_ops[30] = 0.00000000
c_ops[31] = 0.00000000
c_ops[32] = 0.00000000
c_ops[33] = 0.00000000
c_ops[34] = 0.00000000
c_ops[35] = 0.00000000
c_ops[36] = 0.00000000
c_ops[37] = 0.00000000
c_ops[38] = 0.00000000
c_ops[39] = 0.00000000
c_ops[40] = 0.00000000
c_ops[41] = 0.00000000
c_ops[42] = 0.00000000
c_ops[43] = 0.0000000

In [ ]:
np.max(c_op_list[307].full())

(0.009811451159649303+0.005097391489778971j)

In [ ]:
c_op_list[307]

Quantum object: dims = [[100], [100]], shape = (100, 100), type = oper, isherm = False
Qobj data =
[[-4.75190920e-18+9.33828583e-18j -2.66332167e-18+2.49841343e-18j
   1.44544883e-17-7.08700336e-18j ... -2.84887440e-09-4.69337804e-09j
   1.16115139e-19-1.01417800e-19j -1.10087659e-18+1.79531307e-18j]
 [ 5.28678421e-19+6.79496865e-19j  3.39129137e-21-8.14011669e-20j
   2.99665179e-18-3.01261126e-18j ...  3.52659516e-11+2.43551964e-11j
  -5.10365834e-18-3.39548041e-17j  1.01730267e-16+5.76915170e-17j]
 [-1.99671418e-18-3.26206347e-19j -9.17010326e-17-9.91872886e-17j
   2.03943352e-17+7.77672136e-18j ... -2.31641082e-08+4.38530978e-09j
  -4.64113922e-19+4.05276811e-19j -1.28575451e-18-1.56688323e-18j]
 ...
 [-3.51114183e-11+5.78443049e-11j  3.25364602e-12-2.24701686e-12j
   2.45471970e-10+4.64714905e-11j ...  1.37315836e-19+1.68597718e-19j
  -7.96081258e-09+6.11333671e-09j  6.91070279e-12-8.12243519e-11j]
 [-4.20338049e-20+9.61714482e-21j -2.40517107e-20+1.98594737e-19j
  -5.38376696e-20+

In [ ]:
=

SyntaxError: invalid syntax (1763773627.py, line 1)

In [ ]:
arg_select = [H_drive_select, W_20_50, num_cpus, c_op_list, logi_idx_select, 
            option_ideal, option_noisy]

f_noise = Parallel(n_jobs=n_job)(delayed(ut.cz_fidelity_log_noise)
                                    (args_indep, *arg_select)
                                for args_indep in params)
ut.print_data(f'f_{t1_tphi_other}us_{n_truc}', f_noise, num_digits=8)

In [ ]:
=

SyntaxError: invalid syntax (1763773627.py, line 1)